# Run 2 — Full teleportation tomography on the free instance

**What this is:** the hardware version of the tomography protocol in
your `hardware_style_tomography.csv`. Each of the six states is
teleported, and Bob's qubit is measured in **X, Y and Z** bases —
**1,024 shots per basis, 3,072 per state, 18 circuits** — then Bob's
state is reconstructed (Bloch vector → density matrix → fidelity).

**Your simulator benchmark to beat/compare:** mean fidelity **0.903**
(range 0.884–0.915), sitting a consistent **≈0.021 below** the exact
noisy fidelity 0.925. The open questions this run answers: what does
*real* hardware give, and does the reconstruction bias hold?

**Cost: FREE.** This runs on your **Open (free) instance** against
**ibm_kingston** (156-qubit Heron r2) — a different Heron than fez, so
you also get a machine-to-machine comparison. Your free instance has
10 minutes of QPU time untouched; this run should use well under that
(estimate only — the job page shows actual usage after).

**Safety:** `MODE` starts as `"test"` (local simulator — ideal
fidelities print ≈1.000, which proves the analysis code). Flip to
`"hardware"` only with a working personal API key. The key goes into a
hidden prompt and is **never saved** in this notebook.

In [ ]:
# Colab does not ship with Qiskit in this session -- install it first.
# This takes about a minute. Run this cell, then use Runtime > Run all.
%pip install -q qiskit==2.6.1 qiskit-aer==0.18.2 qiskit-ibm-runtime==0.51.0
print("Packages ready: qiskit 2.6.1, qiskit-aer, qiskit-ibm-runtime.")

In [ ]:
# "test"     = local simulator (free, no account) -- run this first
# "hardware" = real ibm_kingston on the FREE Open instance
MODE = "test"

BACKEND_NAME = "ibm_kingston"  # Heron r2, 156 qubits (free instance)
SHOTS = 1024                   # per basis -> 3072 per state, as in your CSV

# Free Open instance: paste its CRN here if the platform asks for one,
# or leave "" and the platform default is used.
INSTANCE_CRN = ""
print("MODE:", MODE, "| backend:", BACKEND_NAME, "| shots/basis:", SHOTS)

In [ ]:
if MODE == "hardware":
    import getpass
    from qiskit_ibm_runtime import QiskitRuntimeService

    token = getpass.getpass("Paste your IBM Cloud API key (hidden): ")
    kwargs = dict(channel="ibm_quantum_platform", token=token)
    if INSTANCE_CRN:
        kwargs["instance"] = INSTANCE_CRN
    service = QiskitRuntimeService(**kwargs)
    backend = service.backend(BACKEND_NAME)
    print("Connected:", backend.name, "-", backend.num_qubits, "qubits")
else:
    from qiskit_aer import AerSimulator
    backend = AerSimulator()
    print("TEST MODE: local simulator, no account, no cost.")

## Six states × three bases = 18 teleportation circuits

In [ ]:
import numpy as np

# The six standard test states (same set as the Lab Pack notebook; the
# tomography dataset in the pack uses |-i> as the sixth state).
PREPS = {
    "|0>":  lambda c: None,
    "|1>":  lambda c: c.x(0),
    "|+>":  lambda c: c.h(0),
    "|->":  lambda c: (c.x(0), c.h(0)),
    "|+i>": lambda c: (c.h(0), c.s(0)),
    "|-i>": lambda c: (c.h(0), c.sdg(0)),
}
STATES = list(PREPS.keys())

# Theory: Bob's Z-basis probability of measuring 0 for each input state.
THEORY_P0 = {"|0>": 1.0, "|1>": 0.0, "|+>": 0.5, "|->": 0.5,
             "|+i>": 0.5, "|-i>": 0.5}
print("States:", STATES)

In [ ]:
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister

BASES = ["X", "Y", "Z"]
# Target Bloch vectors of the six pure states
TARGET_BLOCH = {"|0>": (0, 0, 1), "|1>": (0, 0, -1), "|+>": (1, 0, 0),
                "|->": (-1, 0, 0), "|+i>": (0, 1, 0), "|-i>": (0, -1, 0)}


def teleportation_circuit(prep, basis="Z"):
    """Same dynamic teleportation circuit as the Oct 19 job, with Bob's
    readout rotated into the requested basis for tomography."""
    qr = QuantumRegister(3, "q")
    alice0 = ClassicalRegister(1, "alice0")
    alice1 = ClassicalRegister(1, "alice1")
    bob = ClassicalRegister(1, "bob")
    qc = QuantumCircuit(qr, alice0, alice1, bob)

    prep(qc)
    qc.barrier()
    qc.h(1); qc.cx(1, 2)
    qc.barrier()
    qc.cx(0, 1); qc.h(0)
    qc.measure(0, alice0[0])
    qc.measure(1, alice1[0])
    qc.barrier()
    with qc.if_test((alice1[0], 1)):
        qc.x(2)
    with qc.if_test((alice0[0], 1)):
        qc.z(2)
    qc.barrier()
    if basis == "X":
        qc.h(2)
    elif basis == "Y":
        qc.sdg(2); qc.h(2)
    qc.measure(2, bob[0])
    return qc


plan = [(s, b) for s in STATES for b in BASES]   # 18 (state, basis) pairs
circuits = [teleportation_circuit(PREPS[s], basis=b) for s, b in plan]
print(f"Built {len(circuits)} circuits: {len(STATES)} states x {len(BASES)} bases.")

## Run it

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuits = pm.run(circuits)

if MODE == "hardware":
    from qiskit_ibm_runtime import SamplerV2 as RuntimeSampler
    sampler = RuntimeSampler(mode=backend)
else:
    from qiskit.primitives import BackendSamplerV2
    sampler = BackendSamplerV2(backend=backend)

pubs = [(isa,) for isa in isa_circuits]
job_or_result = sampler.run(pubs, shots=SHOTS)
result = job_or_result.result() if hasattr(job_or_result, "result") else job_or_result
if MODE == "hardware":
    print("Job ID:", job_or_result.job_id())
print(f"Got results for {len(result)} PUBs at {SHOTS} shots each.")

## Reconstruct Bob's state and compute fidelity

From each basis: expectation = (n0 − n1) / shots. The three expectations
are Bob's Bloch vector **r**. For a pure target with Bloch vector **t**,
fidelity = (1 + **r**·**t**) / 2. Results are saved as a CSV in the
same spirit as your simulator tomography file.

In [ ]:
import csv

expect = {}   # (state, basis) -> expectation value
for (state, basis), pub_result in zip(plan, result):
    counts = pub_result.data.bob.get_counts()
    expect[(state, basis)] = (counts.get("0", 0) - counts.get("1", 0)) / SHOTS

rows = []
print(f"{'State':<7}{'<X>':>7}{'<Y>':>7}{'<Z>':>7}{'Fidelity':>10}")
for state in STATES:
    r = np.array([expect[(state, "X")], expect[(state, "Y")], expect[(state, "Z")]])
    t = np.array(TARGET_BLOCH[state], dtype=float)
    fidelity = float(0.5 * (1.0 + r @ t))
    rows.append({"state": state, "expect_x": float(r[0]), "expect_y": float(r[1]),
                 "expect_z": float(r[2]), "fidelity": fidelity})
    print(f"{state:<7}{r[0]:>7.3f}{r[1]:>7.3f}{r[2]:>7.3f}{fidelity:>10.4f}")

mean_f = float(np.mean([row["fidelity"] for row in rows]))
print(f"\nMean fidelity: {mean_f:.4f}")
print("Simulator benchmark from your files: 0.903 (exact noisy: 0.925).")
print("In TEST mode expect ~1.000 -- anything lower on hardware is the")
print("real machine's noise signature, which is the result we want.")

with open("hardware_tomography_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader(); writer.writerows(rows)
print("Saved hardware_tomography_results.csv")

## When this finishes on hardware

Send me the saved CSV (or a screenshot of the table). We'll compare
state-by-state against your 0.903 simulator benchmark, check whether
the ≈ −0.021 reconstruction bias survives on real hardware, and fold
the numbers into the Hardware Edition of the Lab Pack.